# Amazon ML — Business Entity Resolution (Kaggle, Restart-Safe)

This notebook is a Kaggle-ready rewrite of the supplied pipeline.

### Design goals
- **Restart-safe:** every expensive stage writes a checkpoint to `/kaggle/working/ber_checkpoints`.
- **Low RAM:** target Source-2/3 files are streamed in chunks; we never load the full target universe.
- **Precision-first:** threshold is tuned for **F0.5**, including **macro F0.5** and positive-class F0.5.
- **Higher ceiling:** exact normalized matches and strong deterministic rules are used before the ML score.
- **Better negatives:** training mines hard negatives from the same blocking neighborhoods instead of only random negatives.
- **Submission-safe:** creates `matching_results.tsv` and `candidate_pairs.tsv` with the required headers.

> **Important:** no notebook can honestly guarantee a leaderboard score above 0.98 without seeing the hidden test labels. This version is optimized toward that target and validates aggressively on held-out training entities.


In [1]:
# 0. Kaggle setup
!pip install -q rapidfuzz
import os, re, gc, time, json, math, hashlib, pickle, warnings
from pathlib import Path
from collections import defaultdict, Counter

import numpy as np
import pandas as pd
from rapidfuzz.fuzz import ratio, token_set_ratio, WRatio
from sklearn.ensemble import ExtraTreesClassifier, RandomForestClassifier
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import precision_score, recall_score, fbeta_score, confusion_matrix
import joblib

warnings.filterwarnings("ignore")

# Kaggle's attached dataset can have different mount names.
CANDIDATE_ROOTS = [
    Path("/kaggle/input"),
    Path("/kaggle/working"),
    Path("."),
]

def find_file(filename):
    hits = []
    for root in CANDIDATE_ROOTS:
        if root.exists():
            try:
                hits.extend(root.rglob(filename))
            except Exception:
                pass
    if not hits:
        raise FileNotFoundError(
            f"Could not find {filename}. Attach the competition dataset to the Kaggle notebook."
        )
    # Prefer a path containing train/test and not a duplicate working copy.
    hits = sorted(set(map(str, hits)), key=lambda p: ("/kaggle/input" not in p, len(p)))
    return Path(hits[0])

CKPT = Path("/kaggle/working/ber_checkpoints")
OUT = Path("/kaggle/working/output")
CKPT.mkdir(parents=True, exist_ok=True)
OUT.mkdir(parents=True, exist_ok=True)

print("Checkpoint directory:", CKPT)
print("Output directory:", OUT)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 27.6 MB/s eta 0:00:00
Checkpoint directory: /kaggle/working/ber_checkpoints
Output directory: /kaggle/working/output


In [2]:
# 1. Locate the competition files
FILES = {}
for name in [
    "train_source1.tsv", "train_source2.tsv", "train_source3.tsv",
    "train_ground_truth.tsv",
    "test_source1.tsv", "test_source2.tsv", "test_source3.tsv"
]:
    try:
        FILES[name] = find_file(name)
        print(f"{name:25s} -> {FILES[name]}")
    except FileNotFoundError:
        print(f"{name:25s} -> NOT FOUND")

required = ["train_source1.tsv", "train_ground_truth.tsv",
            "test_source1.tsv", "test_source2.tsv", "test_source3.tsv"]
missing = [x for x in required if x not in FILES]
if missing:
    raise FileNotFoundError("Missing required files: " + ", ".join(missing))


train_source1.tsv         -> /kaggle/input/datasets/satwiksps/amazon-ml-challenge-2026/dataset/train/train_source1.tsv
train_source2.tsv         -> /kaggle/input/datasets/satwiksps/amazon-ml-challenge-2026/dataset/train/train_source2.tsv
train_source3.tsv         -> /kaggle/input/datasets/satwiksps/amazon-ml-challenge-2026/dataset/train/train_source3.tsv
train_ground_truth.tsv    -> /kaggle/input/datasets/satwiksps/amazon-ml-challenge-2026/dataset/train/train_ground_truth.tsv
test_source1.tsv          -> /kaggle/input/datasets/satwiksps/amazon-ml-challenge-2026/dataset/test/test_source1.tsv
test_source2.tsv          -> /kaggle/input/datasets/satwiksps/amazon-ml-challenge-2026/dataset/test/test_source2.tsv
test_source3.tsv          -> /kaggle/input/datasets/satwiksps/amazon-ml-challenge-2026/dataset/test/test_source3.tsv


In [3]:
# 2. Normalization + blocking + similarity features
STOPWORDS = {
    "the","and","of","for","a","an","inc","incorporated","llc","ltd","limited",
    "corp","corporation","company","co","private","pvt","plc","llp"
}

def normalize_text(x):
    if x is None or (isinstance(x, float) and pd.isna(x)):
        return ""
    import unicodedata
    x = str(x).lower()
    x = unicodedata.normalize("NFKD", x)
    x = re.sub(r"[^\w\s]", " ", x, flags=re.UNICODE)
    return re.sub(r"\s+", " ", x).strip()

def normalize_name(x):
    x = normalize_text(x)
    for pattern, sub in {
        r"\bprivate limited\b":"pvt ltd", r"\bprivate\b":"pvt",
        r"\blimited\b":"ltd", r"\bcorporation\b":"corp",
        r"\bcompany\b":"co", r"\bincorporated\b":"inc"
    }.items():
        x = re.sub(pattern, sub, x)
    return x.strip()

def normalize_address(x):
    x = normalize_text(x)
    for pattern, sub in {
        r"\broad\b":"rd", r"\bstreet\b":"st", r"\bavenue\b":"ave",
        r"\bboulevard\b":"blvd", r"\blane\b":"ln", r"\bbuilding\b":"bldg",
        r"\bapartment\b":"apt", r"\bfloor\b":"flr", r"\bnumber\b":"no"
    }.items():
        x = re.sub(pattern, sub, x)
    return x.strip()

def meaningful_words(x):
    return [w for w in normalize_text(x).split() if len(w) >= 3 and w not in STOPWORDS]

def get_blocks(name, address, country):
    country = normalize_text(country)
    if not country:
        return set()
    nw, aw = meaningful_words(name), meaningful_words(address)
    full = normalize_text(name)
    blocks = set()
    if full:
        blocks.add(country + "_EXACT_" + full)
    if not nw:
        return blocks
    first, last = nw[0], nw[-1]
    blocks.add(f"{country}_FL_{first[:6]}_{last[:6]}")
    if len(nw) >= 2:
        w1, w2 = nw[0], nw[1]
        blocks.add(f"{country}_N2_{w1[:6]}_{w2[:6]}")
        blocks.add(f"{country}_N2_{w2[:6]}_{w1[:6]}")
    if aw:
        af = aw[0]
        blocks.add(f"{country}_NA_{first[:5]}_{af[:6]}")
        blocks.add(f"{country}_NA_{last[:5]}_{af[:6]}")
    if len(aw) >= 2:
        blocks.add(f"{country}_AA_{aw[0][:5]}_{aw[-1][:5]}")
    return blocks

def jaccard(a, b):
    sa, sb = set(a.split()), set(b.split())
    return len(sa & sb) / len(sa | sb) if sa and sb else 0.0

FEATURES = [
    "name_ratio","name_token_ratio","name_wratio","name_jaccard",
    "address_ratio","address_token_ratio","address_wratio","address_jaccard",
    "country_same","name_exact","address_exact",
    "name_addr_avg","strong_exact"
]

def make_features(a, b):
    n1,n2 = a["name_norm"],b["name_norm"]
    ad1,ad2 = a["address_norm"],b["address_norm"]
    c1,c2 = a["country_norm"],b["country_norm"]
    nr, ar = ratio(n1,n2)/100, ratio(ad1,ad2)/100
    nt, at = token_set_ratio(n1,n2)/100, token_set_ratio(ad1,ad2)/100
    nw, aw = WRatio(n1,n2)/100, WRatio(ad1,ad2)/100
    ne = int(bool(n1) and n1 == n2)
    ae = int(bool(ad1) and ad1 == ad2)
    cs = int(bool(c1) and c1 == c2)
    return {
        "name_ratio":nr, "name_token_ratio":nt, "name_wratio":nw, "name_jaccard":jaccard(n1,n2),
        "address_ratio":ar, "address_token_ratio":at, "address_wratio":aw, "address_jaccard":jaccard(ad1,ad2),
        "country_same":cs, "name_exact":ne, "address_exact":ae,
        "name_addr_avg":(nr+ar)/2,
        "strong_exact":int(ne and cs or ae and cs)
    }

def norm_record(row):
    return {
        "name_norm": normalize_name(row.business_name),
        "address_norm": normalize_address(row.business_address),
        "country_norm": normalize_text(row.country)
    }

print("Feature count:", len(FEATURES))


Feature count: 13


In [4]:
# 3. Restart-safe helpers
def save_pickle(obj, path):
    path = Path(path)
    tmp = path.with_suffix(path.suffix + ".tmp")
    with open(tmp, "wb") as f:
        pickle.dump(obj, f, protocol=pickle.HIGHEST_PROTOCOL)
    os.replace(tmp, path)

def load_pickle(path):
    with open(path, "rb") as f:
        return pickle.load(f)

def checkpoint(name, obj):
    path = CKPT / name
    save_pickle(obj, path)
    print("Checkpoint saved:", path)
    return path

def exists(name):
    return (CKPT / name).exists()

print("Existing checkpoints:", [p.name for p in CKPT.iterdir()][:20])


Existing checkpoints: []


In [5]:
# 4. Load training Source-1 + ground truth
if exists("train_base.pkl"):
    train_base = load_pickle(CKPT/"train_base.pkl")
    s1 = train_base["s1"]
    truth = train_base["truth"]
    print("Resumed training base.")
else:
    s1 = pd.read_csv(FILES["train_source1.tsv"], sep="\t", encoding="latin1", dtype=str).fillna("")
    gt = pd.read_csv(FILES["train_ground_truth.tsv"], sep="\t", encoding="latin1", dtype=str).fillna("")
    truth = {}
    for r in gt.itertuples(index=False):
        value = "" if pd.isna(r.matched_entity_ids) else str(r.matched_entity_ids)
        truth[r.source1_entity_id] = {x.strip() for x in value.split(",") if x.strip()}
    train_base = {"s1": s1, "truth": truth}
    checkpoint("train_base.pkl", train_base)
    del gt
    gc.collect()

print("S1:", len(s1), "Ground-truth S1 keys:", len(truth))


Checkpoint saved: /kaggle/working/ber_checkpoints/train_base.pkl
S1: 2206821 Ground-truth S1 keys: 2206821


In [6]:
# 5. Sample training entities
# Balanced matched/unmatched S1 sample. Increase to 60k if Kaggle runtime allows.
S1_SAMPLE_SIZE = 60000
if exists("s1_sample_ids.pkl"):
    sampled_ids = load_pickle(CKPT/"s1_sample_ids.pkl")
else:
    matched_ids = [eid for eid in s1.entity_id if truth.get(eid)]
    unmatched_ids = [eid for eid in s1.entity_id if not truth.get(eid)]
    rng = np.random.default_rng(42)
    nm = min(S1_SAMPLE_SIZE//2, len(matched_ids))
    nu = min(S1_SAMPLE_SIZE-nm, len(unmatched_ids))
    sampled_ids = list(rng.choice(matched_ids, nm, replace=False)) + list(rng.choice(unmatched_ids, nu, replace=False))
    checkpoint("s1_sample_ids.pkl", sampled_ids)

s1s = s1[s1.entity_id.isin(sampled_ids)].copy()
print("Sampled S1:", len(s1s))


Checkpoint saved: /kaggle/working/ber_checkpoints/s1_sample_ids.pkl
Sampled S1: 60000


In [7]:
# 6. Build compact S1 block index
if exists("s1_block_index.pkl"):
    block_to_s1 = load_pickle(CKPT/"s1_block_index.pkl")
else:
    block_to_s1 = defaultdict(list)
    for r in s1s.itertuples(index=False):
        for k in get_blocks(r.business_name, r.business_address, r.country):
            # Keep only moderately selective blocks.
            if len(block_to_s1[k]) < 100:
                block_to_s1[k].append(r.entity_id)
    block_to_s1 = dict(block_to_s1)
    checkpoint("s1_block_index.pkl", block_to_s1)

print("Block keys:", len(block_to_s1))


Checkpoint saved: /kaggle/working/ber_checkpoints/s1_block_index.pkl
Block keys: 345210


In [8]:
# 7. Mine hard negatives + load only required target rows
# We scan target files in chunks. This is deliberately disk/RAM bounded.
NEGATIVES_PER_S1 = 10
MAX_TARGET_LINES = 2_000_000  # raise if runtime permits
if exists("training_pairs.pkl"):
    pack = load_pickle(CKPT/"training_pairs.pkl")
    X = pd.DataFrame(pack["X"])
    y = np.asarray(pack["y"])
    groups = pack["groups"]
    print("Resumed training pairs:", len(X))
else:
    negative_ids = defaultdict(list)
    positive_ids = set()
    for eid in s1s.entity_id:
        positive_ids.update(truth.get(eid, set()))

    needed = set(positive_ids)
    scanned = 0
    target_records = {}

    for fp in [FILES.get("train_source2.tsv"), FILES.get("train_source3.tsv")]:
        if fp is None or not fp.exists() or scanned >= MAX_TARGET_LINES:
            continue
        for chunk in pd.read_csv(fp, sep="\t", encoding="latin1", dtype=str, chunksize=100_000):
            chunk = chunk.fillna("")
            for r in chunk.itertuples(index=False):
                tid = r.entity_id
                hits = set()
                for k in get_blocks(r.business_name,r.business_address,r.country):
                    ids = block_to_s1.get(k)
                    if ids:
                        hits.update(ids)
                if hits:
                    target_records[tid] = r
                    for sid in hits:
                        if tid not in truth.get(sid,set()) and len(negative_ids[sid]) < NEGATIVES_PER_S1:
                            negative_ids[sid].append(tid)
                            needed.add(tid)
            scanned += len(chunk)
            print(f"Scanned {scanned:,} target rows")
            del chunk
            gc.collect()
            if scanned >= MAX_TARGET_LINES:
                break

    # Make sure all positive target records are available.
    if len(target_records) < len(needed):
        for fp in [FILES.get("train_source2.tsv"), FILES.get("train_source3.tsv")]:
            if fp is None or not fp.exists() or len(target_records) >= len(needed):
                continue
            for chunk in pd.read_csv(fp, sep="\t", encoding="latin1", dtype=str,
                                    chunksize=100_000).fillna(""):
                sel = chunk[chunk.entity_id.isin(needed)]
                for r in sel.itertuples(index=False):
                    target_records[r.entity_id] = r
                if len(target_records) >= len(needed):
                    break

    X_rows, y_rows, groups = [], [], []
    for r in s1s.itertuples(index=False):
        sid = r.entity_id
        a = norm_record(r)
        for tid in truth.get(sid,set()):
            if tid in target_records:
                X_rows.append(make_features(a,norm_record(target_records[tid])))
                y_rows.append(1); groups.append(sid)
        for tid in negative_ids[sid]:
            if tid in target_records:
                X_rows.append(make_features(a,norm_record(target_records[tid])))
                y_rows.append(0); groups.append(sid)

    X = pd.DataFrame(X_rows).reindex(columns=FEATURES, fill_value=0)
    y = np.asarray(y_rows, dtype=np.int8)
    checkpoint("training_pairs.pkl", {"X":X.to_dict("list"),"y":y,"groups":groups})
    del target_records, negative_ids, X_rows
    gc.collect()

print("Training pairs:", len(X), "positive:", int(y.sum()), "negative:", int((y==0).sum()))


Scanned 100,000 target rows
Scanned 200,000 target rows
Scanned 300,000 target rows
Scanned 400,000 target rows
Scanned 500,000 target rows
Scanned 600,000 target rows
Scanned 700,000 target rows
Scanned 800,000 target rows
Scanned 900,000 target rows
Scanned 1,000,000 target rows
Scanned 1,100,000 target rows
Scanned 1,200,000 target rows
Scanned 1,300,000 target rows
Scanned 1,400,000 target rows
Scanned 1,500,000 target rows
Scanned 1,600,000 target rows
Scanned 1,700,000 target rows
Scanned 1,800,000 target rows
Scanned 1,900,000 target rows
Scanned 2,000,000 target rows
Checkpoint saved: /kaggle/working/ber_checkpoints/training_pairs.pkl
Training pairs: 443335 positive: 17651 negative: 425684


In [9]:
# 8. Train + fine-grained threshold search for macro F0.5
if exists("model_bundle.pkl"):
    bundle = load_pickle(CKPT/"model_bundle.pkl")
    model = bundle["model"]
    threshold = bundle["threshold"]
    print("Resumed model. Threshold:", threshold)
else:
    splitter = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=42)
    tr, va = next(splitter.split(X,y,groups=groups))
    Xtr, Xva, ytr, yva = X.iloc[tr], X.iloc[va], y[tr], y[va]

    # ExtraTrees is fast on small dense feature sets and handles nonlinear interactions.
    model = ExtraTreesClassifier(
        n_estimators=500,
        max_depth=24,
        min_samples_leaf=1,
        max_features="sqrt",
        class_weight="balanced",
        random_state=42,
        n_jobs=-1
    )
    t0=time.time()
    model.fit(Xtr,ytr)
    print("Model fit seconds:", round(time.time()-t0,1))

    probs = model.predict_proba(Xva)[:,1]
    best = None
    for t in np.arange(0.20,0.996,0.005):
        pred=(probs>=t).astype(np.int8)
        fpos=fbeta_score(yva,pred,beta=0.5,zero_division=0)
        fmacro=fbeta_score(yva,pred,beta=0.5,average="macro",zero_division=0)
        prec=precision_score(yva,pred,zero_division=0)
        rec=recall_score(yva,pred,zero_division=0)
        score=(fmacro,fpos,prec)
        if best is None or score>best[0]:
            best=(score,float(t),prec,rec,fmacro,fpos)

    _, threshold, prec, rec, fmacro, fpos = best
    bundle={"model":model,"threshold":threshold,"feature_names":FEATURES}
    checkpoint("model_bundle.pkl",bundle)

print(f"Chosen threshold={threshold:.3f} | validation precision={prec:.5f} recall={rec:.5f} macro-F0.5={fmacro:.5f} positive-F0.5={fpos:.5f}")


Model fit seconds: 44.2
Checkpoint saved: /kaggle/working/ber_checkpoints/model_bundle.pkl
Chosen threshold=0.920 | validation precision=0.97872 recall=0.64589 macro-F0.5=0.93772 positive-F0.5=0.88728


In [10]:
# 9. Optional threshold diagnostics
# If you want maximum precision, use the table below rather than blindly trusting 0.5.
if 'Xva' in globals():
    rows=[]
    for t in np.arange(0.50,0.996,0.01):
        p=(probs>=t).astype(np.int8)
        rows.append({
            "threshold":round(float(t),3),
            "precision":precision_score(yva,p,zero_division=0),
            "recall":recall_score(yva,p,zero_division=0),
            "f0.5_positive":fbeta_score(yva,p,beta=.5,zero_division=0),
            "f0.5_macro":fbeta_score(yva,p,beta=.5,average="macro",zero_division=0)
        })
    display(pd.DataFrame(rows).sort_values("f0.5_macro",ascending=False).head(15))


,threshold,precision,recall,f0.5_positive,f0.5_macro
42,0.92,0.978723,0.645886,0.887277,0.937721
43,0.93,0.982548,0.632407,0.884594,0.936171
44,0.94,0.983936,0.619208,0.880240,0.933783
45,0.95,0.986703,0.604325,0.875865,0.931362
34,0.84,0.908101,0.749228,0.871155,0.931090
37,0.87,0.919499,0.721707,0.871718,0.930973
38,0.88,0.922489,0.711879,0.870954,0.930444
33,0.83,0.903291,0.755406,0.869256,0.930221
36,0.86,0.913563,0.730132,0.869856,0.930155
35,0.85,0.909343,0.737995,0.868990,0.929832


In [11]:
# 10. Export the trained model in the original format too
model_data={"model":model,"best_threshold":float(threshold),"feature_names":FEATURES}
joblib.dump(model_data, OUT/"model.joblib", compress=3)
print("Saved:", OUT/"model.joblib")


Saved: /kaggle/working/output/model.joblib


In [12]:
# 11. Restart-safe prediction: build S1 block index
# IMPORTANT: this stage is checkpointed separately from the training stages.
if exists("test_s1_index.pkl"):
    test_pack=load_pickle(CKPT/"test_s1_index.pkl")
    test_s1_raw=test_pack["raw"]
    test_block_to_s1=test_pack["blocks"]
    print("Resumed test S1 index.")
else:
    test_s1 = pd.read_csv(FILES["test_source1.tsv"],sep="\t",encoding="latin1",dtype=str).fillna("")
    test_s1_raw={}
    test_block_to_s1=defaultdict(list)
    for r in test_s1.itertuples(index=False):
        sid=r.entity_id
        test_s1_raw[sid]=(r.business_name,r.business_address,r.country)
        for k in get_blocks(r.business_name,r.business_address,r.country):
            # Prevent giant generic blocks from exploding candidate count.
            if len(test_block_to_s1[k]) < 50:
                test_block_to_s1[k].append(sid)
    test_block_to_s1=dict(test_block_to_s1)
    checkpoint("test_s1_index.pkl",{"raw":test_s1_raw,"blocks":test_block_to_s1})
    del test_s1
    gc.collect()

print("Test S1:",len(test_s1_raw),"block keys:",len(test_block_to_s1))


Checkpoint saved: /kaggle/working/ber_checkpoints/test_s1_index.pkl
Test S1: 1732544 block keys: 6226111


In [13]:
# ============================================================
# STEP 1 — KAGGLE SETUP
# ============================================================

import os
import re
import gc
import time
import json
import math
import pickle
import warnings
import zipfile
from pathlib import Path
from collections import defaultdict, Counter

import numpy as np
import pandas as pd

from rapidfuzz.fuzz import ratio, token_set_ratio, WRatio

from sklearn.ensemble import ExtraTreesClassifier
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import (
    precision_score,
    recall_score,
    fbeta_score,
    confusion_matrix
)

import joblib

warnings.filterwarnings("ignore")

print("Environment loaded.")

Environment loaded.


In [14]:
# ============================================================
# STEP 2 — LOCATE COMPETITION FILES
# ============================================================

CANDIDATE_ROOTS = [
    Path("/kaggle/input"),
    Path("/kaggle/working"),
    Path(".")
]

def find_file(filename):

    hits = []

    for root in CANDIDATE_ROOTS:

        if root.exists():

            try:
                hits.extend(root.rglob(filename))
            except Exception:
                pass

    if not hits:

        raise FileNotFoundError(
            f"Could not find {filename}. "
            "Attach the competition dataset to Kaggle."
        )

    hits = sorted(
        set(map(str, hits)),
        key=lambda p: (
            "/kaggle/input" not in p,
            len(p)
        )
    )

    return Path(hits[0])


FILES = {}

file_names = [
    "train_source1.tsv",
    "train_source2.tsv",
    "train_source3.tsv",
    "train_ground_truth.tsv",
    "test_source1.tsv",
    "test_source2.tsv",
    "test_source3.tsv"
]

for name in file_names:

    try:

        FILES[name] = find_file(name)

        print(
            f"{name:25s} -> {FILES[name]}"
        )

    except FileNotFoundError:

        print(
            f"{name:25s} -> NOT FOUND"
        )


required = [
    "train_source1.tsv",
    "train_ground_truth.tsv",
    "test_source1.tsv",
    "test_source2.tsv",
    "test_source3.tsv"
]

missing = [
    x for x in required
    if x not in FILES
]

if missing:

    raise FileNotFoundError(
        "Missing required files: "
        + ", ".join(missing)
    )

print("\nAll required files found.")

train_source1.tsv         -> /kaggle/input/datasets/satwiksps/amazon-ml-challenge-2026/dataset/train/train_source1.tsv
train_source2.tsv         -> /kaggle/input/datasets/satwiksps/amazon-ml-challenge-2026/dataset/train/train_source2.tsv
train_source3.tsv         -> /kaggle/input/datasets/satwiksps/amazon-ml-challenge-2026/dataset/train/train_source3.tsv
train_ground_truth.tsv    -> /kaggle/input/datasets/satwiksps/amazon-ml-challenge-2026/dataset/train/train_ground_truth.tsv
test_source1.tsv          -> /kaggle/input/datasets/satwiksps/amazon-ml-challenge-2026/dataset/test/test_source1.tsv
test_source2.tsv          -> /kaggle/input/datasets/satwiksps/amazon-ml-challenge-2026/dataset/test/test_source2.tsv
test_source3.tsv          -> /kaggle/input/datasets/satwiksps/amazon-ml-challenge-2026/dataset/test/test_source3.tsv

All required files found.


In [15]:
# ============================================================
# STEP 3 — CHECKPOINT DIRECTORIES
# ============================================================

CKPT = Path(
    "/kaggle/working/ber_checkpoints"
)

OUT = Path(
    "/kaggle/working/output"
)

CKPT.mkdir(
    parents=True,
    exist_ok=True
)

OUT.mkdir(
    parents=True,
    exist_ok=True
)

# IMPORTANT:
# New prediction directory so we don't mix the old
# 40-candidate run with this emergency 8-candidate run.

PRED_DIR = (
    CKPT /
    "prediction_chunks_fast"
)

PRED_DIR.mkdir(
    parents=True,
    exist_ok=True
)

print("Checkpoint:", CKPT)
print("Output:", OUT)
print("Prediction chunks:", PRED_DIR)

Checkpoint: /kaggle/working/ber_checkpoints
Output: /kaggle/working/output
Prediction chunks: /kaggle/working/ber_checkpoints/prediction_chunks_fast


In [16]:
# ============================================================
# STEP 4 — NORMALIZATION + BLOCKING + FEATURES
# ============================================================

STOPWORDS = {
    "the",
    "and",
    "of",
    "for",
    "a",
    "an",
    "inc",
    "incorporated",
    "llc",
    "ltd",
    "limited",
    "corp",
    "corporation",
    "company",
    "co",
    "private",
    "pvt",
    "plc",
    "llp"
}


def normalize_text(x):

    if x is None:
        return ""

    if isinstance(x, float) and pd.isna(x):
        return ""

    import unicodedata

    x = str(x).lower()

    x = unicodedata.normalize(
        "NFKD",
        x
    )

    x = re.sub(
        r"[^\w\s]",
        " ",
        x,
        flags=re.UNICODE
    )

    x = re.sub(
        r"\s+",
        " ",
        x
    )

    return x.strip()


def normalize_name(x):

    x = normalize_text(x)

    replacements = {
        r"\bprivate limited\b": "pvt ltd",
        r"\bprivate\b": "pvt",
        r"\blimited\b": "ltd",
        r"\bcorporation\b": "corp",
        r"\bcompany\b": "co",
        r"\bincorporated\b": "inc"
    }

    for pattern, sub in replacements.items():

        x = re.sub(
            pattern,
            sub,
            x
        )

    return x.strip()


def normalize_address(x):

    x = normalize_text(x)

    replacements = {
        r"\broad\b": "rd",
        r"\bstreet\b": "st",
        r"\bavenue\b": "ave",
        r"\bboulevard\b": "blvd",
        r"\blane\b": "ln",
        r"\bbuilding\b": "bldg",
        r"\bapartment\b": "apt",
        r"\bfloor\b": "flr",
        r"\bnumber\b": "no"
    }

    for pattern, sub in replacements.items():

        x = re.sub(
            pattern,
            sub,
            x
        )

    return x.strip()


def meaningful_words(x):

    return [
        w
        for w in normalize_text(x).split()
        if len(w) >= 3
        and w not in STOPWORDS
    ]


def get_blocks(
    name,
    address,
    country
):

    country = normalize_text(country)

    if not country:
        return set()

    nw = meaningful_words(name)
    aw = meaningful_words(address)

    full = normalize_text(name)

    blocks = set()

    if full:

        blocks.add(
            country
            + "_EXACT_"
            + full
        )

    if not nw:

        return blocks

    first = nw[0]
    last = nw[-1]

    blocks.add(
        f"{country}_FL_"
        f"{first[:6]}_"
        f"{last[:6]}"
    )

    if len(nw) >= 2:

        w1 = nw[0]
        w2 = nw[1]

        blocks.add(
            f"{country}_N2_"
            f"{w1[:6]}_"
            f"{w2[:6]}"
        )

        blocks.add(
            f"{country}_N2_"
            f"{w2[:6]}_"
            f"{w1[:6]}"
        )

    if aw:

        af = aw[0]

        blocks.add(
            f"{country}_NA_"
            f"{first[:5]}_"
            f"{af[:6]}"
        )

        blocks.add(
            f"{country}_NA_"
            f"{last[:5]}_"
            f"{af[:6]}"
        )

    if len(aw) >= 2:

        blocks.add(
            f"{country}_AA_"
            f"{aw[0][:5]}_"
            f"{aw[-1][:5]}"
        )

    return blocks


def jaccard(a, b):

    sa = set(a.split())
    sb = set(b.split())

    if not sa or not sb:

        return 0.0

    return (
        len(sa & sb)
        /
        len(sa | sb)
    )


FEATURES = [
    "name_ratio",
    "name_token_ratio",
    "name_wratio",
    "name_jaccard",
    "address_ratio",
    "address_token_ratio",
    "address_wratio",
    "address_jaccard",
    "country_same",
    "name_exact",
    "address_exact",
    "name_addr_avg",
    "strong_exact"
]


def make_features(a, b):

    n1 = a["name_norm"]
    n2 = b["name_norm"]

    ad1 = a["address_norm"]
    ad2 = b["address_norm"]

    c1 = a["country_norm"]
    c2 = b["country_norm"]

    nr = ratio(n1, n2) / 100
    ar = ratio(ad1, ad2) / 100

    nt = token_set_ratio(n1, n2) / 100
    at = token_set_ratio(ad1, ad2) / 100

    nw = WRatio(n1, n2) / 100
    aw = WRatio(ad1, ad2) / 100

    ne = int(
        bool(n1)
        and
        n1 == n2
    )

    ae = int(
        bool(ad1)
        and
        ad1 == ad2
    )

    cs = int(
        bool(c1)
        and
        c1 == c2
    )

    return {

        "name_ratio": nr,

        "name_token_ratio": nt,

        "name_wratio": nw,

        "name_jaccard":
            jaccard(n1, n2),

        "address_ratio": ar,

        "address_token_ratio": at,

        "address_wratio": aw,

        "address_jaccard":
            jaccard(ad1, ad2),

        "country_same": cs,

        "name_exact": ne,

        "address_exact": ae,

        "name_addr_avg":
            (nr + ar) / 2,

        "strong_exact":
            int(
                (ne and cs)
                or
                (ae and cs)
            )
    }


def norm_record(row):

    return {

        "name_norm":
            normalize_name(
                row.business_name
            ),

        "address_norm":
            normalize_address(
                row.business_address
            ),

        "country_norm":
            normalize_text(
                row.country
            )
    }


print(
    "Features:",
    len(FEATURES)
)

Features: 13


In [17]:
# ============================================================
# STEP 5 — CHECKPOINT HELPERS
# ============================================================

def save_pickle(
    obj,
    path
):

    path = Path(path)

    tmp = Path(
        str(path) + ".tmp"
    )

    with open(
        tmp,
        "wb"
    ) as f:

        pickle.dump(
            obj,
            f,
            protocol=pickle.HIGHEST_PROTOCOL
        )

    os.replace(
        tmp,
        path
    )


def load_pickle(path):

    with open(
        path,
        "rb"
    ) as f:

        return pickle.load(f)


def checkpoint(
    name,
    obj
):

    path = CKPT / name

    save_pickle(
        obj,
        path
    )

    print(
        "Checkpoint saved:",
        path
    )

    return path


def exists(name):

    return (
        CKPT / name
    ).exists()


print(
    "Existing checkpoints:",
    [
        p.name
        for p in CKPT.iterdir()
    ][:30]
)

Existing checkpoints: ['training_pairs.pkl', 'train_base.pkl', 's1_sample_ids.pkl', 'test_s1_index.pkl', 's1_block_index.pkl', 'model_bundle.pkl', 'prediction_chunks_fast']


In [18]:
# ============================================================
# STEP 6 — LOAD TRAINING SOURCE-1 + GROUND TRUTH
# ============================================================

if exists(
    "train_base.pkl"
):

    train_base = load_pickle(
        CKPT / "train_base.pkl"
    )

    s1 = train_base["s1"]
    truth = train_base["truth"]

    print(
        "Resumed training base."
    )

else:

    s1 = pd.read_csv(
        FILES["train_source1.tsv"],
        sep="\t",
        encoding="latin1",
        dtype=str
    ).fillna("")

    gt = pd.read_csv(
        FILES["train_ground_truth.tsv"],
        sep="\t",
        encoding="latin1",
        dtype=str
    ).fillna("")

    truth = {}

    for r in gt.itertuples(
        index=False
    ):

        value = (
            ""
            if pd.isna(
                r.matched_entity_ids
            )
            else
            str(
                r.matched_entity_ids
            )
        )

        truth[
            r.source1_entity_id
        ] = {
            x.strip()
            for x in value.split(",")
            if x.strip()
        }

    checkpoint(
        "train_base.pkl",
        {
            "s1": s1,
            "truth": truth
        }
    )

    del gt

    gc.collect()


print(
    "S1:",
    len(s1)
)

print(
    "Ground truth keys:",
    len(truth)
)

Resumed training base.
S1: 2206821
Ground truth keys: 2206821


In [19]:
# ============================================================
# STEP 7 — SAMPLE TRAINING ENTITIES
# ============================================================

S1_SAMPLE_SIZE = 60_000

if exists(
    "s1_sample_ids.pkl"
):

    sampled_ids = load_pickle(
        CKPT / "s1_sample_ids.pkl"
    )

    print(
        "Loaded existing sample."
    )

else:

    matched_ids = [
        eid
        for eid in s1.entity_id
        if truth.get(eid)
    ]

    unmatched_ids = [
        eid
        for eid in s1.entity_id
        if not truth.get(eid)
    ]

    rng = np.random.default_rng(
        42
    )

    nm = min(
        S1_SAMPLE_SIZE // 2,
        len(matched_ids)
    )

    nu = min(
        S1_SAMPLE_SIZE - nm,
        len(unmatched_ids)
    )

    sampled_ids = (
        list(
            rng.choice(
                matched_ids,
                nm,
                replace=False
            )
        )
        +
        list(
            rng.choice(
                unmatched_ids,
                nu,
                replace=False
            )
        )
    )

    checkpoint(
        "s1_sample_ids.pkl",
        sampled_ids
    )


s1s = s1[
    s1.entity_id.isin(
        sampled_ids
    )
].copy()


print(
    "Sampled S1:",
    len(s1s)
)

Loaded existing sample.
Sampled S1: 60000


In [20]:
# ============================================================
# STEP 8 — BUILD TRAINING BLOCK INDEX
# ============================================================

if exists(
    "s1_block_index.pkl"
):

    block_to_s1 = load_pickle(
        CKPT / "s1_block_index.pkl"
    )

    print(
        "Loaded training block index."
    )

else:

    block_to_s1 = defaultdict(list)

    for r in s1s.itertuples(
        index=False
    ):

        for k in get_blocks(
            r.business_name,
            r.business_address,
            r.country
        ):

            if len(
                block_to_s1[k]
            ) < 100:

                block_to_s1[k].append(
                    r.entity_id
                )

    block_to_s1 = dict(
        block_to_s1
    )

    checkpoint(
        "s1_block_index.pkl",
        block_to_s1
    )


print(
    "Block keys:",
    len(block_to_s1)
)

Loaded training block index.
Block keys: 345210


In [21]:
# ============================================================
# STEP 9 — MINE HARD NEGATIVES
# ============================================================

NEGATIVES_PER_S1 = 10
MAX_TARGET_LINES = 2_000_000


if exists(
    "training_pairs.pkl"
):

    pack = load_pickle(
        CKPT / "training_pairs.pkl"
    )

    X = pd.DataFrame(
        pack["X"]
    )

    y = np.asarray(
        pack["y"]
    )

    groups = pack["groups"]

    print(
        "Resumed training pairs:",
        len(X)
    )

else:

    negative_ids = defaultdict(list)

    positive_ids = set()

    for eid in s1s.entity_id:

        positive_ids.update(
            truth.get(
                eid,
                set()
            )
        )


    needed = set(
        positive_ids
    )

    scanned = 0

    target_records = {}


    for fp in [
        FILES.get(
            "train_source2.tsv"
        ),
        FILES.get(
            "train_source3.tsv"
        )
    ]:

        if (
            fp is None
            or
            not fp.exists()
            or
            scanned >= MAX_TARGET_LINES
        ):

            continue


        for chunk in pd.read_csv(
            fp,
            sep="\t",
            encoding="latin1",
            dtype=str,
            chunksize=100_000
        ).fillna(""):

            for r in chunk.itertuples(
                index=False
            ):

                tid = r.entity_id

                hits = set()

                for k in get_blocks(
                    r.business_name,
                    r.business_address,
                    r.country
                ):

                    ids = block_to_s1.get(k)

                    if ids:

                        hits.update(
                            ids
                        )


                if hits:

                    target_records[
                        tid
                    ] = r


                    for sid in hits:

                        if (
                            tid
                            not in
                            truth.get(
                                sid,
                                set()
                            )
                            and
                            len(
                                negative_ids[sid]
                            )
                            <
                            NEGATIVES_PER_S1
                        ):

                            negative_ids[
                                sid
                            ].append(
                                tid
                            )

                            needed.add(
                                tid
                            )


            scanned += len(
                chunk
            )

            print(
                f"Scanned {scanned:,} "
                f"target rows"
            )

            del chunk

            gc.collect()


            if (
                scanned
                >=
                MAX_TARGET_LINES
            ):

                break


    # Make sure positive records exist
    if (
        len(target_records)
        <
        len(needed)
    ):

        for fp in [
            FILES.get(
                "train_source2.tsv"
            ),
            FILES.get(
                "train_source3.tsv"
            )
        ]:

            if (
                fp is None
                or
                not fp.exists()
                or
                len(target_records)
                >=
                len(needed)
            ):

                continue


            for chunk in pd.read_csv(
                fp,
                sep="\t",
                encoding="latin1",
                dtype=str,
                chunksize=100_000
            ).fillna(""):

                sel = chunk[
                    chunk.entity_id.isin(
                        needed
                    )
                ]

                for r in sel.itertuples(
                    index=False
                ):

                    target_records[
                        r.entity_id
                    ] = r


                if (
                    len(target_records)
                    >=
                    len(needed)
                ):

                    break


    X_rows = []
    y_rows = []
    groups = []


    for r in s1s.itertuples(
        index=False
    ):

        sid = r.entity_id

        a = norm_record(r)


        # positives

        for tid in truth.get(
            sid,
            set()
        ):

            if tid in target_records:

                X_rows.append(
                    make_features(
                        a,
                        norm_record(
                            target_records[tid]
                        )
                    )
                )

                y_rows.append(1)

                groups.append(sid)


        # hard negatives

        for tid in negative_ids[sid]:

            if tid in target_records:

                X_rows.append(
                    make_features(
                        a,
                        norm_record(
                            target_records[tid]
                        )
                    )
                )

                y_rows.append(0)

                groups.append(sid)


    X = pd.DataFrame(
        X_rows
    ).reindex(
        columns=FEATURES,
        fill_value=0
    )

    y = np.asarray(
        y_rows,
        dtype=np.int8
    )


    checkpoint(
        "training_pairs.pkl",
        {
            "X":
                X.to_dict("list"),

            "y":
                y,

            "groups":
                groups
        }
    )


    del target_records
    del negative_ids
    del X_rows

    gc.collect()


print(
    "Training pairs:",
    len(X)
)

print(
    "Positive:",
    int(y.sum())
)

print(
    "Negative:",
    int((y == 0).sum())
)

Resumed training pairs: 443335
Training pairs: 443335
Positive: 17651
Negative: 425684


In [22]:
# ============================================================
# STEP 10 — TRAIN MODEL + OPTIMIZE F0.5 THRESHOLD
# ============================================================

if exists(
    "model_bundle.pkl"
):

    bundle = load_pickle(
        CKPT / "model_bundle.pkl"
    )

    model = bundle["model"]

    threshold = bundle["threshold"]

    print(
        "Loaded existing model."
    )

    print(
        "Threshold:",
        threshold
    )

else:

    splitter = GroupShuffleSplit(
        n_splits=1,
        test_size=0.20,
        random_state=42
    )

    tr, va = next(
        splitter.split(
            X,
            y,
            groups=groups
        )
    )


    Xtr = X.iloc[tr]
    Xva = X.iloc[va]

    ytr = y[tr]
    yva = y[va]


    model = ExtraTreesClassifier(

        n_estimators=500,

        max_depth=24,

        min_samples_leaf=1,

        max_features="sqrt",

        class_weight="balanced",

        random_state=42,

        n_jobs=-1
    )


    t0 = time.time()

    model.fit(
        Xtr,
        ytr
    )

    print(
        "Model training seconds:",
        round(
            time.time() - t0,
            1
        )
    )


    probs = model.predict_proba(
        Xva
    )[:, 1]


    best = None


    # Fine threshold search
    for t in np.arange(
        0.20,
        0.996,
        0.005
    ):

        pred = (
            probs >= t
        ).astype(
            np.int8
        )


        fpos = fbeta_score(
            yva,
            pred,
            beta=0.5,
            zero_division=0
        )


        fmacro = fbeta_score(
            yva,
            pred,
            beta=0.5,
            average="macro",
            zero_division=0
        )


        prec = precision_score(
            yva,
            pred,
            zero_division=0
        )


        rec = recall_score(
            yva,
            pred,
            zero_division=0
        )


        score = (
            fmacro,
            fpos,
            prec
        )


        if (
            best is None
            or
            score > best[0]
        ):

            best = (
                score,
                float(t),
                prec,
                rec,
                fmacro,
                fpos
            )


    (
        _,
        threshold,
        prec,
        rec,
        fmacro,
        fpos
    ) = best


    checkpoint(
        "model_bundle.pkl",
        {
            "model":
                model,

            "threshold":
                threshold,

            "feature_names":
                FEATURES
        }
    )


print(
    f"Threshold={threshold:.3f}"
)

# These variables may not exist after loading a checkpoint,
# so only print validation metrics if available.

if "fmacro" in globals():

    print(
        f"Validation precision={prec:.5f}"
    )

    print(
        f"Validation recall={rec:.5f}"
    )

    print(
        f"Validation macro-F0.5={fmacro:.5f}"
    )

    print(
        f"Validation positive-F0.5={fpos:.5f}"
    )

Loaded existing model.
Threshold: 0.9200000000000006
Threshold=0.920
Validation precision=0.97872
Validation recall=0.64589
Validation macro-F0.5=0.93772
Validation positive-F0.5=0.88728


In [23]:
# ============================================================
# STEP 11 — EXPORT MODEL
# ============================================================

model_data = {
    "model": model,
    "best_threshold": float(threshold),
    "feature_names": FEATURES
}

joblib.dump(
    model_data,
    OUT / "model.joblib",
    compress=3
)

print(
    "Saved:",
    OUT / "model.joblib"
)

Saved: /kaggle/working/output/model.joblib


In [24]:
# ============================================================
# STEP 12 — BUILD TEST SOURCE-1 INDEX
# ============================================================

if exists(
    "test_s1_index.pkl"
):

    test_pack = load_pickle(
        CKPT / "test_s1_index.pkl"
    )

    test_s1_raw = test_pack[
        "raw"
    ]

    test_block_to_s1 = test_pack[
        "blocks"
    ]

    print(
        "Resumed test S1 index."
    )

else:

    test_s1 = pd.read_csv(
        FILES["test_source1.tsv"],
        sep="\t",
        encoding="latin1",
        dtype=str
    ).fillna("")


    test_s1_raw = {}

    test_block_to_s1 = defaultdict(list)


    for r in test_s1.itertuples(
        index=False
    ):

        sid = r.entity_id

        test_s1_raw[sid] = (
            r.business_name,
            r.business_address,
            r.country
        )


        for k in get_blocks(
            r.business_name,
            r.business_address,
            r.country
        ):

            # Keep blocks bounded
            if (
                len(
                    test_block_to_s1[k]
                )
                <
                50
            ):

                test_block_to_s1[
                    k
                ].append(
                    sid
                )


    test_block_to_s1 = dict(
        test_block_to_s1
    )


    checkpoint(
        "test_s1_index.pkl",
        {
            "raw":
                test_s1_raw,

            "blocks":
                test_block_to_s1
        }
    )


    del test_s1

    gc.collect()


print(
    "Test S1:",
    len(test_s1_raw)
)

print(
    "Test block keys:",
    len(test_block_to_s1)
)

Resumed test S1 index.
Test S1: 1732544
Test block keys: 6226111


In [25]:
# ============================================================
# STEP 13 — PRE-NORMALIZE TEST SOURCE-1
# ============================================================

TEST_S1_NORM_FILE = (
    CKPT /
    "test_s1_norm.pkl"
)


if TEST_S1_NORM_FILE.exists():

    test_s1_norm = load_pickle(
        TEST_S1_NORM_FILE
    )

    print(
        "Loaded normalized test S1."
    )

else:

    print(
        "Normalizing test S1 once..."
    )

    test_s1_norm = {}

    for i, (
        sid,
        raw
    ) in enumerate(
        test_s1_raw.items()
    ):

        test_s1_norm[sid] = {

            "name_norm":
                normalize_name(
                    raw[0]
                ),

            "address_norm":
                normalize_address(
                    raw[1]
                ),

            "country_norm":
                normalize_text(
                    raw[2]
                )
        }


        if (
            i + 1
        ) % 100_000 == 0:

            print(
                "Normalized:",
                i + 1
            )


    save_pickle(
        test_s1_norm,
        TEST_S1_NORM_FILE
    )


print(
    "Normalized S1:",
    len(test_s1_norm)
)

Normalizing test S1 once...
Normalized: 100000
Normalized: 200000
Normalized: 300000
Normalized: 400000
Normalized: 500000
Normalized: 600000
Normalized: 700000
Normalized: 800000
Normalized: 900000
Normalized: 1000000
Normalized: 1100000
Normalized: 1200000
Normalized: 1300000
Normalized: 1400000
Normalized: 1500000
Normalized: 1600000
Normalized: 1700000
Normalized S1: 1732544


In [26]:
# ============================================================
# STEP 14 — FAST, RESTART-SAFE PREDICTION
# ============================================================

FAST_MAX_CANDIDATES = 8

MODEL_BATCH_SIZE = 8192

CHUNK_SIZE = 50_000


print(
    "FAST MODE"
)

print(
    "Candidates per test row:",
    FAST_MAX_CANDIDATES
)

print(
    "Model batch size:",
    MODEL_BATCH_SIZE
)

print(
    "Threshold:",
    threshold
)

FAST MODE
Candidates per test row: 8
Model batch size: 8192
Threshold: 0.9200000000000006


In [27]:
# ============================================================
# STEP 15 — SAFE CHUNK WRITING
# ============================================================

def save_prediction_chunk(
    chunk_file,
    candidate_rows,
    match_rows
):

    tmp_file = Path(
        str(chunk_file) + ".tmp"
    )


    payload = {

        "candidates":
            candidate_rows,

        "matches":
            match_rows
    }


    with open(
        tmp_file,
        "wb"
    ) as f:

        pickle.dump(
            payload,
            f,
            protocol=pickle.HIGHEST_PROTOCOL
        )


    os.replace(
        tmp_file,
        chunk_file
    )

In [28]:
# ============================================================
# STEP 16 — PROCESS ONE CHUNK
# ============================================================

def process_fast_chunk(
    chunk,
    source_name,
    chunk_no
):

    chunk_id = (
        f"{source_name}__"
        f"{chunk_no:06d}"
    )


    chunk_file = (
        PRED_DIR /
        f"{chunk_id}.pkl"
    )


    # --------------------------------------------------------
    # RESTART SAFETY
    # --------------------------------------------------------

    if chunk_file.exists():

        print(
            f"[SKIP] "
            f"{source_name} "
            f"chunk {chunk_no}"
        )

        return False


    chunk = chunk.fillna("")


    candidate_records = []


    # ========================================================
    # A. CANDIDATE GENERATION
    # ========================================================

    for r in chunk.itertuples(
        index=False
    ):

        tid = r.entity_id


        # Normalize target once
        b = norm_record(r)


        keys = get_blocks(
            r.business_name,
            r.business_address,
            r.country
        )


        hits = set()


        for k in keys:

            ids = test_block_to_s1.get(
                k
            )

            if ids:

                hits.update(
                    ids
                )


        if not hits:

            continue


        ranked = []


        # ====================================================
        # Cheap candidate ranking
        # ====================================================

        for sid in hits:

            a = test_s1_norm.get(
                sid
            )

            if a is None:

                continue


            target_name = b[
                "name_norm"
            ]

            source_name_norm = a[
                "name_norm"
            ]


            target_country = b[
                "country_norm"
            ]

            source_country = a[
                "country_norm"
            ]


            # -----------------------------------------------
            # Exact checks
            # -----------------------------------------------

            name_exact = (
                bool(target_name)
                and
                target_name
                ==
                source_name_norm
            )


            country_same = (
                bool(target_country)
                and
                target_country
                ==
                source_country
            )


            # -----------------------------------------------
            # Cheap name similarity
            # -----------------------------------------------

            if name_exact:

                name_score = 100.0

            elif (
                target_name
                and
                source_name_norm
            ):

                name_score = ratio(
                    source_name_norm,
                    target_name
                )

            else:

                name_score = 0.0


            # -----------------------------------------------
            # Address similarity
            #
            # Only calculate it when name has reasonable
            # similarity. This saves time.
            # -----------------------------------------------

            if (
                name_score >= 45
                and
                a["address_norm"]
                and
                b["address_norm"]
            ):

                address_score = ratio(
                    a["address_norm"],
                    b["address_norm"]
                )

            else:

                address_score = 0.0


            # -----------------------------------------------
            # Candidate ranking
            # -----------------------------------------------

            rank = (
                name_score * 0.70
                +
                address_score * 0.25
                +
                (
                    5.0
                    if country_same
                    else 0.0
                )
            )


            # Exact name + country is extremely important

            if (
                name_exact
                and
                country_same
            ):

                rank += 1000


            ranked.append(
                (
                    rank,
                    sid
                )
            )


        # ====================================================
        # Keep ONLY top candidates
        # ====================================================

        ranked.sort(
            key=lambda x: x[0],
            reverse=True
        )


        selected = ranked[
            :FAST_MAX_CANDIDATES
        ]


        for _, sid in selected:

            candidate_records.append(
                (
                    sid,
                    tid,
                    b
                )
            )


    # ========================================================
    # B. NO CANDIDATES
    # ========================================================

    if not candidate_records:

        save_prediction_chunk(
            chunk_file,
            [],
            []
        )

        print(
            f"[DONE] "
            f"{source_name} "
            f"chunk={chunk_no} "
            f"| candidates=0 "
            f"| matches=0"
        )

        return True


    # ========================================================
    # C. FEATURE GENERATION
    # ========================================================

    feature_rows = []

    pair_ids = []


    for sid, tid, b in candidate_records:

        a = test_s1_norm[sid]


        feat = make_features(
            a,
            b
        )


        feature_rows.append(
            feat
        )


        pair_ids.append(
            (
                sid,
                tid
            )
        )


    del candidate_records

    gc.collect()


    # ========================================================
    # D. BATCH MODEL PREDICTION
    # ========================================================

    probabilities = np.empty(
        len(feature_rows),
        dtype=np.float32
    )


    for start in range(
        0,
        len(feature_rows),
        MODEL_BATCH_SIZE
    ):

        end = min(
            start
            +
            MODEL_BATCH_SIZE,
            len(feature_rows)
        )


        X_batch = pd.DataFrame(
            feature_rows[
                start:end
            ]
        ).reindex(
            columns=FEATURES,
            fill_value=0
        )


        probabilities[
            start:end
        ] = model.predict_proba(
            X_batch
        )[:, 1]


        del X_batch


    # ========================================================
    # E. CREATE RESULTS
    # ========================================================

    candidate_rows = []

    match_rows = []


    for i, (
        (sid, tid),
        prob
    ) in enumerate(
        zip(
            pair_ids,
            probabilities
        )
    ):

        feat = feature_rows[i]


        strong = (

            feat["strong_exact"] == 1

            or

            (
                feat["name_exact"] == 1
                and
                feat["country_same"] == 1
            )

            or

            (
                feat["name_ratio"] >= 0.985
                and
                feat["address_ratio"] >= 0.90
                and
                feat["country_same"] == 1
            )

            or

            (
                feat["name_ratio"] >= 0.995
                and
                feat["country_same"] == 1
            )
        )


        candidate_rows.append(
            (
                sid,
                tid,
                float(prob),
                int(strong)
            )
        )


        if (
            strong
            or
            prob >= threshold
        ):

            match_rows.append(
                (
                    sid,
                    tid,
                    float(prob),
                    int(strong)
                )
            )


    # ========================================================
    # F. WRITE TO DISK
    # ========================================================

    save_prediction_chunk(
        chunk_file,
        candidate_rows,
        match_rows
    )


    print(
        f"[DONE] "
        f"{source_name} "
        f"chunk={chunk_no} "
        f"| candidates="
        f"{len(candidate_rows):,} "
        f"| matches="
        f"{len(match_rows):,}"
    )


    # ========================================================
    # G. MEMORY CLEANUP
    # ========================================================

    del feature_rows
    del pair_ids
    del probabilities
    del candidate_rows
    del match_rows

    gc.collect()


    return True

In [29]:
# ============================================================
# STEP 17 — STREAM BOTH TEST SOURCES
# ============================================================

for fp in [
    FILES["test_source2.tsv"],
    FILES["test_source3.tsv"]
]:

    source_name = Path(fp).stem


    print("\n")
    print("=" * 75)
    print(
        "PROCESSING:",
        source_name
    )
    print("=" * 75)


    for chunk_no, chunk in enumerate(
        pd.read_csv(
            fp,
            sep="\t",
            encoding="latin1",
            dtype=str,
            chunksize=CHUNK_SIZE
        )
    ):


        chunk_file = (
            PRED_DIR /
            f"{source_name}__"
            f"{chunk_no:06d}.pkl"
        )


        # ----------------------------------------------------
        # Resume support
        # ----------------------------------------------------

        if chunk_file.exists():

            if chunk_no % 10 == 0:

                print(
                    f"[SKIP EXISTING] "
                    f"{source_name} "
                    f"chunk={chunk_no}"
                )

            del chunk

            continue


        # ----------------------------------------------------
        # Process new chunk
        # ----------------------------------------------------

        process_fast_chunk(
            chunk,
            source_name,
            chunk_no
        )


        del chunk

        gc.collect()


    print(
        "\nCOMPLETED:",
        source_name
    )


print("\n")
print("=" * 75)
print("ALL TEST SOURCES COMPLETE")
print("=" * 75)



PROCESSING: test_source2
[DONE] test_source2 chunk=0 | candidates=255,657 | matches=32,611
[DONE] test_source2 chunk=1 | candidates=254,917 | matches=33,207
[DONE] test_source2 chunk=2 | candidates=255,996 | matches=33,470
[DONE] test_source2 chunk=3 | candidates=255,052 | matches=32,701
[DONE] test_source2 chunk=4 | candidates=254,743 | matches=33,251
[DONE] test_source2 chunk=5 | candidates=254,893 | matches=33,121
[DONE] test_source2 chunk=6 | candidates=255,941 | matches=33,203
[DONE] test_source2 chunk=7 | candidates=255,010 | matches=32,535
[DONE] test_source2 chunk=8 | candidates=255,252 | matches=33,092
[DONE] test_source2 chunk=9 | candidates=254,899 | matches=32,856
[DONE] test_source2 chunk=10 | candidates=254,537 | matches=32,494
[DONE] test_source2 chunk=11 | candidates=256,529 | matches=33,246
[DONE] test_source2 chunk=12 | candidates=254,917 | matches=33,019
[DONE] test_source2 chunk=13 | candidates=255,658 | matches=33,155
[DONE] test_source2 chunk=14 | candidates=255

In [30]:
# ============================================================
# STEP 18 — CHECK COMPLETED CHUNKS
# ============================================================

chunk_files = sorted(
    PRED_DIR.glob("*.pkl")
)

print(
    "Prediction chunks:",
    len(chunk_files)
)

for fp in chunk_files[:5]:

    print(
        fp.name
    )

print("...")

for fp in chunk_files[-5:]:

    print(
        fp.name
    )

Prediction chunks: 200
test_source2__000000.pkl
test_source2__000001.pkl
test_source2__000002.pkl
test_source2__000003.pkl
test_source2__000004.pkl
...
test_source3__000097.pkl
test_source3__000098.pkl
test_source3__000099.pkl
test_source3__000100.pkl
test_source3__000101.pkl


In [31]:
# ============================================================
# STEP 19 — AGGREGATE PREDICTION CHUNKS
# ============================================================

candidate_map = defaultdict(list)

match_map = defaultdict(list)


chunk_files = sorted(
    PRED_DIR.glob("*.pkl")
)


print(
    "Chunks found:",
    len(chunk_files)
)


for i, fp in enumerate(
    chunk_files
):

    with open(
        fp,
        "rb"
    ) as f:

        data = pickle.load(f)


    for sid, tid, prob, strong in data[
        "candidates"
    ]:

        candidate_map[sid].append(
            (
                tid,
                prob,
                strong
            )
        )


    for sid, tid, prob, strong in data[
        "matches"
    ]:

        match_map[sid].append(
            (
                tid,
                prob,
                strong
            )
        )


    del data


    if (
        i + 1
    ) % 25 == 0:

        print(
            f"Loaded "
            f"{i+1}/"
            f"{len(chunk_files)} chunks"
        )


    gc.collect()


print(
    "Candidate S1 keys:",
    len(candidate_map)
)

print(
    "Match S1 keys:",
    len(match_map)
)

Chunks found: 200
Loaded 25/200 chunks
Loaded 50/200 chunks
Loaded 75/200 chunks
Loaded 100/200 chunks
Loaded 125/200 chunks
Loaded 150/200 chunks
Loaded 175/200 chunks
Loaded 200/200 chunks
Candidate S1 keys: 1726338
Match S1 keys: 1500159


In [32]:
# ============================================================
# STEP 20 — FINALIZE TSV FILES
# ============================================================

MAX_CANDIDATES_PER_S1 = 40
MAX_MATCHES_PER_S1 = 10


s1_ids = list(
    test_s1_raw.keys()
)


candidate_path = (
    OUT /
    "candidate_pairs.tsv"
)

matching_path = (
    OUT /
    "matching_results.tsv"
)


with open(
    candidate_path,
    "w",
    encoding="utf-8"
) as fc, open(
    matching_path,
    "w",
    encoding="utf-8"
) as fm:


    fc.write(
        "source1_entity_id\t"
        "candidate_entity_ids\n"
    )


    fm.write(
        "source1_entity_id\t"
        "matched_entity_ids\n"
    )


    for sid in s1_ids:


        # ----------------------------------------------------
        # Candidates
        # ----------------------------------------------------

        candidates = {}


        for (
            tid,
            score,
            strong
        ) in candidate_map.get(
            sid,
            []
        ):

            old = candidates.get(
                tid
            )

            if (
                old is None
                or
                score > old
            ):

                candidates[
                    tid
                ] = score


        candidate_list = sorted(
            candidates.items(),
            key=lambda x: x[1],
            reverse=True
        )[
            :MAX_CANDIDATES_PER_S1
        ]


        candidate_ids = [
            tid
            for tid, _
            in candidate_list
        ]


        # ----------------------------------------------------
        # Matches
        # ----------------------------------------------------

        matches = {}


        for (
            tid,
            score,
            strong
        ) in match_map.get(
            sid,
            []
        ):

            old = matches.get(
                tid
            )


            if (
                old is None
                or
                (
                    strong,
                    score
                )
                >
                (
                    old[0],
                    old[1]
                )
            ):

                matches[
                    tid
                ] = (
                    strong,
                    score
                )


        match_list = sorted(
            matches.items(),
            key=lambda x: (
                x[1][0],
                x[1][1]
            ),
            reverse=True
        )[
            :MAX_MATCHES_PER_S1
        ]


        match_ids = [
            tid
            for tid, _
            in match_list
        ]


        # ----------------------------------------------------
        # Write
        # ----------------------------------------------------

        fc.write(
            f"{sid}\t"
            f"{','.join(candidate_ids)}\n"
        )


        fm.write(
            f"{sid}\t"
            f"{','.join(match_ids)}\n"
        )


print(
    "Created:"
)

print(
    candidate_path
)

print(
    matching_path
)

Created:
/kaggle/working/output/candidate_pairs.tsv
/kaggle/working/output/matching_results.tsv


In [33]:
# ============================================================
# STEP 21 — VALIDATE SUBMISSION
# ============================================================

def validate_submission(
    matching_path,
    candidate_path,
    test_source1_path
):

    errors = []


    required = set(
        pd.read_csv(
            test_source1_path,
            sep="\t",
            usecols=[0],
            dtype=str
        ).iloc[:, 0]
    )


    def read_map(
        path,
        second_col
    ):

        df = pd.read_csv(
            path,
            sep="\t",
            dtype=str,
            keep_default_na=False
        )


        expected = [
            "source1_entity_id",
            second_col
        ]


        if list(
            df.columns
        ) != expected:

            errors.append(
                f"{path}: "
                f"wrong header "
                f"{list(df.columns)}"
            )


        if (
            df.source1_entity_id
            .duplicated()
            .any()
        ):

            errors.append(
                f"{path}: "
                "duplicate S1 rows"
            )


        got = set(
            df.source1_entity_id
        )


        if got - required:

            errors.append(
                f"{path}: "
                "unknown S1 IDs"
            )


        if required - got:

            errors.append(
                f"{path}: "
                f"missing S1 IDs: "
                f"{len(required-got)}"
            )


        for v in df[
            second_col
        ]:

            for x in filter(
                None,
                v.split(",")
            ):

                if not x.startswith(
                    (
                        "S2-",
                        "S3-"
                    )
                ):

                    errors.append(
                        f"{path}: "
                        f"invalid target "
                        f"prefix: {x}"
                    )

                    return df


        return df


    m = read_map(
        matching_path,
        "matched_entity_ids"
    )


    c = read_map(
        candidate_path,
        "candidate_entity_ids"
    )


    # Check that every match is a candidate
    if not errors:

        cm = {

            r.source1_entity_id:
                set(
                    filter(
                        None,
                        r.candidate_entity_ids.split(",")
                    )
                )

            for r in c.itertuples(
                index=False
            )
        }


        for r in m.itertuples(
            index=False
        ):

            mids = set(
                filter(
                    None,
                    r.matched_entity_ids.split(",")
                )
            )


            if not (
                mids
                <=
                cm.get(
                    r.source1_entity_id,
                    set()
                )
            ):

                errors.append(
                    "Match outside "
                    "candidate set for "
                    f"{r.source1_entity_id}"
                )

                break


    return errors


errs = validate_submission(
    matching_path,
    candidate_path,
    FILES["test_source1.tsv"]
)


if not errs:

    print(
        "================================"
    )

    print(
        "SUBMISSION VALIDATION PASSED"
    )

    print(
        "================================"
    )

else:

    print(
        "================================"
    )

    print(
        "SUBMISSION VALIDATION ERRORS"
    )

    print(
        "================================"
    )


    for e in errs[:20]:

        print(
            " -",
            e
        )

SUBMISSION VALIDATION ERRORS
 - Match outside candidate set for S1-106407869


In [34]:
# ============================================================
# STEP 22 — SUBMISSION STATISTICS
# ============================================================

submission_df = pd.read_csv(
    matching_path,
    sep="\t",
    dtype=str,
    keep_default_na=False
)


candidate_df = pd.read_csv(
    candidate_path,
    sep="\t",
    dtype=str,
    keep_default_na=False
)


num_s1 = len(
    submission_df
)


num_matched = sum(
    bool(x)
    for x in submission_df[
        "matched_entity_ids"
    ]
)


num_candidates = sum(
    len(
        list(
            filter(
                None,
                x.split(",")
            )
        )
    )
    for x in candidate_df[
        "candidate_entity_ids"
    ]
)


num_matches = sum(
    len(
        list(
            filter(
                None,
                x.split(",")
            )
        )
    )
    for x in submission_df[
        "matched_entity_ids"
    ]
)


print(
    "Source-1 entities:",
    f"{num_s1:,}"
)

print(
    "S1 entities with >=1 match:",
    f"{num_matched:,}"
)

print(
    "Total candidates:",
    f"{num_candidates:,}"
)

print(
    "Total predicted matches:",
    f"{num_matches:,}"
)

print(
    "Match rate:",
    f"{num_matched / max(num_s1,1):.2%}"
)

Source-1 entities: 1,732,544
S1 entities with >=1 match: 1,500,159
Total candidates: 36,123,427
Total predicted matches: 5,481,994
Match rate: 86.59%


In [35]:
# ============================================================
# STEP 23 — CREATE FINAL SUBMISSION ZIP
# ============================================================

submission_zip = (
    OUT /
    "amazon_entity_resolution_submission.zip"
)


with zipfile.ZipFile(
    submission_zip,
    "w",
    zipfile.ZIP_DEFLATED
) as z:

    z.write(
        matching_path,
        "matching_results.tsv"
    )

    z.write(
        candidate_path,
        "candidate_pairs.tsv"
    )

    z.write(
        OUT / "model.joblib",
        "model.joblib"
    )


print(
    "================================"
)

print(
    "FINAL ZIP CREATED"
)

print(
    "================================"
)

print(
    submission_zip
)

print(
    "Size:",
    round(
        submission_zip.stat().st_size
        /
        (1024 * 1024),
        2
    ),
    "MB"
)

FINAL ZIP CREATED
/kaggle/working/output/amazon_entity_resolution_submission.zip
Size: 354.59 MB


In [36]:
# ============================================================
# STEP 24 — FINAL SANITY CHECK
# ============================================================

print("\nFINAL FILES\n")

for fp in [
    candidate_path,
    matching_path,
    OUT / "model.joblib",
    submission_zip
]:

    print(
        fp.name,
        "|",
        round(
            fp.stat().st_size /
            (1024 * 1024),
            2
        ),
        "MB"
    )


print("\nREADY FOR SUBMISSION.")


FINAL FILES

candidate_pairs.tsv | 465.33 MB
matching_results.tsv | 88.9 MB
model.joblib | 120.27 MB
amazon_entity_resolution_submission.zip | 354.59 MB

READY FOR SUBMISSION.
